# Data Cleaning & Preparation

This notebook prepares the ITSM incident dataset for analysis.

Cleaning tasks include:

- Standardizing column names
- Converting timestamp fields
- Checking duplicate ticket IDs
- Validating categorical fields
- Creating time-based features

In [41]:
import pandas as pd
import numpy as np

In [16]:
df = pd.read_csv("../data/raw/itsm_incident_dataset.csv")

df.head()


,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,...,Resolution time,SLA For Resolution,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Latitude,Longitude
0,Closed,TCKT-100000,High,Email,General Inquiry,Security,Khalid Al-Salem,04-07-2024 12.42,04-07-2024 14.42,04-07-2024 13.12,...,04-07-2024 14.30,Met,04-07-2024 14.32,5,Neutral,Cloud,L3,Oman,25.1856,50.9447
1,Closed,TCKT-100001,High,Chat,Network Issue,Customer Service,Ahmed Al-Sabah,23-05-2024 20.03,23-05-2024 22.03,23-05-2024 20.33,...,23-05-2024 22.00,Met,23-05-2024 22.05,4,Dissatisfied,Cloud,L2,Qatar,23.2741,55.3867
2,In Progress,TCKT-100002,Low,Phone,General Inquiry,Development,Mohammed Al-Mansoori,13-04-2024 20.51,14-04-2024 0.51,13-04-2024 21.51,...,14-04-2024 0.47,Met,14-04-2024 0.51,3,Dissatisfied,Software,L1,Bahrain,23.6264,50.1302
3,Resolved,TCKT-100003,Critical,Chat,Access Request,Development,Mohammed Al-Khalifa,13-05-2024 12.50,13-05-2024 13.50,13-05-2024 13.00,...,13-05-2024 13.48,Met,13-05-2024 13.53,5,Dissatisfied,Network,L2,Kuwait,25.0736,54.8437
4,Closed,TCKT-100004,Critical,Portal,Hardware Failure,Customer Service,Hassan Al-Nasser,19-06-2024 22.51,19-06-2024 23.51,19-06-2024 23.01,...,19-06-2024 23.49,Met,19-06-2024 23.54,4,Neutral,Hardware,L3,Qatar,24.7362,51.4839


In [17]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

Index(['status', 'ticket_id', 'priority', 'source', 'topic', 'agent_group',
       'agent_name', 'created_time', 'expected_sla_to_resolve',
       'expected_sla_to_first_response', 'first_response_time',
       'sla_for_first_response', 'resolution_time', 'sla_for_resolution',
       'close_time', 'agent_interactions', 'survey_results', 'product_group',
       'support_level', 'country', 'latitude', 'longitude'],
      dtype='object')

In [18]:
timestamp_cols = [
    "created_time",
    "first_response_time",
    "resolution_time",
    "close_time",
    "expected_sla_to_resolve",
    "expected_sla_to_first_response"
]

for col in timestamp_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df[timestamp_cols].info()

C:\Users\Dhank\AppData\Local\Temp\ipykernel_5272\1742830416.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Dhank\AppData\Local\Temp\ipykernel_5272\1742830416.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Dhank\AppData\Local\Temp\ipykernel_5272\1742830416.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Dhank\AppData\Local\Temp\ipykernel_5272\1742830416.py:12: UserWarning: Could no

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   created_time                    0 non-null      datetime64[ns]
 1   first_response_time             0 non-null      datetime64[ns]
 2   resolution_time                 0 non-null      datetime64[ns]
 3   close_time                      0 non-null      datetime64[ns]
 4   expected_sla_to_resolve         0 non-null      datetime64[ns]
 5   expected_sla_to_first_response  0 non-null      datetime64[ns]
dtypes: datetime64[ns](6)
memory usage: 4.6 MB


In [19]:
df.duplicated(subset="ticket_id").sum()

np.int64(0)

In [20]:
categorical_cols = [
    "status",
    "priority",
    "source",
    "topic",
    "agent_group",
    "support_level"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

In [21]:
df["created_hour"] = df["created_time"].dt.hour

In [22]:
df["created_day_of_week"] = df["created_time"].dt.day_name()

In [23]:
df["created_month"] = df["created_time"].dt.month

In [24]:
df["resolution_duration_hours"] = (
    df["resolution_time"] - df["created_time"]
).dt.total_seconds() / 3600

In [25]:
df[[
    "created_hour",
    "created_day_of_week",
    "created_month",
    "resolution_duration_hours"
]].head()

,created_hour,created_day_of_week,created_month,resolution_duration_hours
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   status                          100000 non-null  object        
 1   ticket_id                       100000 non-null  object        
 2   priority                        100000 non-null  object        
 3   source                          100000 non-null  object        
 4   topic                           100000 non-null  object        
 5   agent_group                     100000 non-null  object        
 6   agent_name                      100000 non-null  object        
 7   created_time                    0 non-null       datetime64[ns]
 8   expected_sla_to_resolve         0 non-null       datetime64[ns]
 9   expected_sla_to_first_response  0 non-null       datetime64[ns]
 10  first_response_time             0 non-null       datetime

In [27]:
df.head()

,status,ticket_id,priority,source,topic,agent_group,agent_name,created_time,expected_sla_to_resolve,expected_sla_to_first_response,...,survey_results,product_group,support_level,country,latitude,longitude,created_hour,created_day_of_week,created_month,resolution_duration_hours
0,Closed,TCKT-100000,High,Email,General Inquiry,Security,Khalid Al-Salem,NaT,NaT,NaT,...,Neutral,Cloud,L3,Oman,25.1856,50.9447,NaN,NaN,NaN,NaN
1,Closed,TCKT-100001,High,Chat,Network Issue,Customer Service,Ahmed Al-Sabah,NaT,NaT,NaT,...,Dissatisfied,Cloud,L2,Qatar,23.2741,55.3867,NaN,NaN,NaN,NaN
2,In Progress,TCKT-100002,Low,Phone,General Inquiry,Development,Mohammed Al-Mansoori,NaT,NaT,NaT,...,Dissatisfied,Software,L1,Bahrain,23.6264,50.1302,NaN,NaN,NaN,NaN
3,Resolved,TCKT-100003,Critical,Chat,Access Request,Development,Mohammed Al-Khalifa,NaT,NaT,NaT,...,Dissatisfied,Network,L2,Kuwait,25.0736,54.8437,NaN,NaN,NaN,NaN
4,Closed,TCKT-100004,Critical,Portal,Hardware Failure,Customer Service,Hassan Al-Nasser,NaT,NaT,NaT,...,Neutral,Hardware,L3,Qatar,24.7362,51.4839,NaN,NaN,NaN,NaN


In [28]:
df["created_time"].head(10)

0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
5   NaT
6   NaT
7   NaT
8   NaT
9   NaT
Name: created_time, dtype: datetime64[ns]

In [30]:
import pandas as pd

df = pd.read_csv("../data/raw/itsm_incident_dataset.csv")

In [31]:
timestamp_cols = [
    "Created time",
    "First response time",
    "Resolution time",
    "Close time",
    "Expected SLA to resolve",
    "Expected SLA to first response"
]

for col in timestamp_cols:
    df[col] = df[col].astype(str).str.replace(r'(\d{2})\.(\d{2})', r'\1:\2', regex=True)

In [32]:
df.head()

,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,...,Resolution time,SLA For Resolution,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Latitude,Longitude
0,Closed,TCKT-100000,High,Email,General Inquiry,Security,Khalid Al-Salem,04-07-2024 12:42,04-07-2024 14:42,04-07-2024 13:12,...,04-07-2024 14:30,Met,04-07-2024 14:32,5,Neutral,Cloud,L3,Oman,25.1856,50.9447
1,Closed,TCKT-100001,High,Chat,Network Issue,Customer Service,Ahmed Al-Sabah,23-05-2024 20:03,23-05-2024 22:03,23-05-2024 20:33,...,23-05-2024 22:00,Met,23-05-2024 22:05,4,Dissatisfied,Cloud,L2,Qatar,23.2741,55.3867
2,In Progress,TCKT-100002,Low,Phone,General Inquiry,Development,Mohammed Al-Mansoori,13-04-2024 20:51,14-04-2024 0.51,13-04-2024 21:51,...,14-04-2024 0.47,Met,14-04-2024 0.51,3,Dissatisfied,Software,L1,Bahrain,23.6264,50.1302
3,Resolved,TCKT-100003,Critical,Chat,Access Request,Development,Mohammed Al-Khalifa,13-05-2024 12:50,13-05-2024 13:50,13-05-2024 13:00,...,13-05-2024 13:48,Met,13-05-2024 13:53,5,Dissatisfied,Network,L2,Kuwait,25.0736,54.8437
4,Closed,TCKT-100004,Critical,Portal,Hardware Failure,Customer Service,Hassan Al-Nasser,19-06-2024 22:51,19-06-2024 23:51,19-06-2024 23:01,...,19-06-2024 23:49,Met,19-06-2024 23:54,4,Neutral,Hardware,L3,Qatar,24.7362,51.4839


In [33]:
for col in timestamp_cols:
    df[col] = pd.to_datetime(df[col], format="%d-%m-%Y %H:%M", errors="coerce")

In [34]:
df[["Created time","Resolution time"]].head()

,Created time,Resolution time
0,2024-07-04 12:42:00,2024-07-04 14:30:00
1,2024-05-23 20:03:00,2024-05-23 22:00:00
2,2024-04-13 20:51:00,NaT
3,2024-05-13 12:50:00,2024-05-13 13:48:00
4,2024-06-19 22:51:00,2024-06-19 23:49:00


In [35]:
df["created_hour"] = df["Created time"].dt.hour
df["created_day_of_week"] = df["Created time"].dt.day_name()
df["created_month"] = df["Created time"].dt.month

df["resolution_duration_hours"] = (
    df["Resolution time"] - df["Created time"]
).dt.total_seconds() / 3600

In [36]:
df[[
    "created_hour",
    "created_day_of_week",
    "created_month",
    "resolution_duration_hours"
]].head()

,created_hour,created_day_of_week,created_month,resolution_duration_hours
0,12.0,Thursday,7.0,1.800000
1,20.0,Thursday,5.0,1.950000
2,20.0,Saturday,4.0,NaN
3,12.0,Monday,5.0,0.966667
4,22.0,Wednesday,6.0,0.966667


In [37]:
df.to_csv("../data/processed/itsm_incident_clean.csv", index=False)

In [43]:
import pandas as pd

df = pd.read_csv("../data/processed/itsm_incident_clean.csv")

In [46]:
(df['resolution_duration_hours'] < 0).sum()

np.int64(0)

In [48]:
df.dtypes

Status                             object
Ticket ID                          object
Priority                           object
Source                             object
Topic                              object
Agent Group                        object
Agent Name                         object
Created time                       object
Expected SLA to resolve            object
Expected SLA to first response     object
First response time                object
SLA For first response             object
Resolution time                    object
SLA For Resolution                 object
Close time                         object
Agent interactions                  int64
Survey results                     object
Product group                      object
Support Level                      object
Country                            object
Latitude                          float64
Longitude                         float64
created_hour                      float64
created_day_of_week               

In [49]:
df[['created_hour','created_day_of_week','created_month']].head()

,created_hour,created_day_of_week,created_month
0,12.0,Thursday,7.0
1,20.0,Thursday,5.0
2,20.0,Saturday,4.0
3,12.0,Monday,5.0
4,22.0,Wednesday,6.0
